In [ ]:
# OmniVoice's own lockfile, from PyTorch's cu128 index for torch; torchvision at
# the version that belongs to that torch so transformers can import beside it.
!pip -q install "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3
!pip -q install "transformers==5.3.0" "accelerate==1.13.0" "webdataset==1.0.2" "safetensors==0.7.0" "tokenizers==0.22.2" "huggingface-hub==1.7.2" "tensorboardX==2.6.4" "soundfile==0.13.1" "librosa==0.11.0" "pydub==0.25.1" 2>&1 | tail -3
!pip -q install --no-deps "git+https://github.com/k2-fsa/OmniVoice@08be0b4ccbac3e13e374e86fbfead4b4cac343e2" 2>&1 | tail -3
!pip -q install speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime 2>&1 | tail -3


In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def captured(command):
    result = subprocess.run(command, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    print(captured([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])[1])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import torch

OFFICIAL = {'torch': '2.8.0+cu128', 'torchaudio': '2.8.0+cu128', 'transformers': '5.3.0', 'accelerate': '1.13.0', 'webdataset': '1.0.2', 'safetensors': '0.7.0', 'tokenizers': '0.22.2', 'huggingface-hub': '1.7.2', 'tensorboardX': '2.6.4', 'soundfile': '0.13.1', 'librosa': '0.11.0', 'pydub': '0.25.1', 'numpy': '2.4.3', 'triton': '3.4.0'}
PINNED = ('torch', 'torchaudio', 'transformers', 'accelerate', 'webdataset', 'safetensors', 'tokenizers', 'huggingface-hub')


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def matches(have, want):
    if have is None:
        return False
    return have == want or (have.split("+")[0] == want.split("+")[0] and "+" not in want)


stack = {name: {"official": want, "installed": installed(name),
                "matches": matches(installed(name), want), "pinned_here": name in PINNED}
         for name, want in OFFICIAL.items()}
direct = json.loads(importlib.metadata.distribution("omnivoice").read_text("direct_url.json") or "{}")
code, driver = captured(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"])
ENVIRONMENT_RECORD = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "bf16_supported_including_emulation": torch.cuda.is_bf16_supported(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "nvidia_smi": driver,
    "official_stack": stack,
    "omnivoice": {"version": installed("omnivoice"),
                  "commit": (direct.get("vcs_info") or {}).get("commit_id"), "url": direct.get("url")},
    "other_packages": {name: installed(name) for name in ("torchvision", "ipykernel", "papermill", "scipy")},
}
Path("/kaggle/working/environment.json").write_text(json.dumps(ENVIRONMENT_RECORD, indent=2))
for name, row in stack.items():
    mark = "ok" if row["matches"] else ("DIFFERS" if row["pinned_here"] else "differs - image's version kept")
    print(f"{name:16} official {row['official']:12} installed {str(row['installed']):16} {mark}")
print("omnivoice", ENVIRONMENT_RECORD["omnivoice"])
print("cuda", torch.version.cuda, "| cudnn", torch.backends.cudnn.version(), "|", driver)
for name in PINNED:
    assert stack[name]["matches"], f"{name} is not the official {OFFICIAL[name]}: {stack[name]['installed']}"
assert ENVIRONMENT_RECORD["omnivoice"]["commit"] == "08be0b4ccbac3e13e374e86fbfead4b4cac343e2", ENVIRONMENT_RECORD["omnivoice"]
assert torch.cuda.device_count() >= 2, "the official recipe trains on two GPUs - use T4 x2"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}, capability {torch.cuda.get_device_capability(i)}, "
          f"{free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk:", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GiB free under /tmp")

INPUT = Path("/kaggle/input")
found = sorted(p for p in INPUT.rglob("repo/scripts/render_frontend_conditions.py") if "persian-voice-frontend-listening" in p.parts)
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo with scripts/render_frontend_conditions.py under /kaggle/input")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
# Kaggle unpacks the .gz when it builds the dataset; either form is checked
# against the hash the split builder recorded for its decompressed content.
MANIFESTS = REPO / "eval/manifests"
SPLITS = next((p for p in (MANIFESTS / "adaptation_splits_v1.jsonl.gz", MANIFESTS / "adaptation_splits_v1.jsonl")
               if p.is_file()), None)
assert SPLITS is not None, "the dataset predates the adaptation splits"
import gzip  # noqa: E402

content = gzip.open(SPLITS).read() if SPLITS.suffix == ".gz" else SPLITS.read_bytes()
expected = json.loads((MANIFESTS / "adaptation_splits_v1.summary.json").read_text())["records_sha256"]
assert hashlib.sha256(content).hexdigest() == expected, "the split file is not the one the builder wrote"
print("splits:", SPLITS.name, "sha256 matches the summary")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-frontend-listening-run-v1"
sys.path.insert(0, str(REPO / "scripts"))
from kernel_gates import Gates, report_group, run_bounded  # noqa: E402

GATES = Gates()
RUNS = {}
LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
WORK = Path("/tmp/adapt")
WORK.mkdir(parents=True, exist_ok=True)
RECORDS = Path("/kaggle/working/records")
RECORDS.mkdir(exist_ok=True)


In [ ]:
from runner_check import check, diagnostics  # noqa: E402

ok, reason, run = check(run_bounded, Path("/kaggle/working/runner_check"), REPO / "scripts/moss15_worker.py")
RUNS["runner_check"] = run.record()
print(run.summary())
seen, line = diagnostics(report_group)
print("descriptor report:", "works" if seen else "DOES NOT WORK", "-", line)
GATES.record("runner", ok, reason)


In [ ]:
def fetch(plan, minutes, flags=()):
    """scripts/fetch_inputs.py under run_bounded: every request timed out, the stage bounded."""
    plan_path, report_path = RECORDS / "fetch_plan.json", RECORDS / "fetch.json"
    plan_path.write_text(json.dumps(plan))
    run = run_bounded([sys.executable, str(REPO / "scripts/fetch_inputs.py"), "--plan", str(plan_path),
                       "--report", str(report_path), *flags], timeout=minutes * 60, log=LOGS / "fetch.log",
                      monitor_every=300)
    RUNS["fetch"] = run.record()
    print(run.summary())
    report = json.loads(report_path.read_text()) if report_path.is_file() else {}
    fetched = {step["name"]: {k: step.get(k) for k in ("gib", "seconds", "tries", "failed_check")}
               for step in report.get("steps", [])}
    return run.ok and report.get("ok") is True, fetched

os.environ["VOICECLONE_OMNIVOICE_PYTHON"] = sys.executable
provenance = found[0].parents[2] / "PROVENANCE.md"
os.environ["VOICECLONE_CODE_COMMIT"] = (provenance.read_text().split("commit ")[-1].strip(" .\n")
                                        if provenance.is_file() else "the dataset's PROVENANCE.md was not found")
NEEDED = ['scripts/render_frontend_conditions.py', 'scripts/ab_omnivoice_ipa.py', 'scripts/omnivoice_worker.py', 'src/voiceclone/text/marks_fa.py', 'eval/corpus/manifest.json', 'eval/corpus/mana_000.wav', 'eval/runs/frontend-listening-v1/texts.json']
missing = [name for name in NEEDED if not (REPO / name).is_file()]
texts = REPO / "eval/runs/frontend-listening-v1/texts.json"
if texts.is_file() and hashlib.sha256(texts.read_bytes()).hexdigest() != "06fd5f36cc1f2cb41a57c105b808dc3dc4dc836f3f01a574925b82f6598e197b":
    missing.append("texts.json is not the registered one")
GATES.record("code", not missing, f"missing or changed: {missing}" if missing else
             "the render script, the marks and the registered texts are present; code from " +
             os.environ["VOICECLONE_CODE_COMMIT"])
if not GATES.skip("download", ("runner", "code")):
    fetched_ok, fetched = fetch({'snapshots': [{'name': 'base', 'repo': 'k2-fsa/OmniVoice', 'revision': 'c5fdb5ccb189668d56333f77ba2629f4cd7535f4', 'local_dir': '/tmp/omnivoice-base', 'sha256': {'audio_tokenizer/model.safetensors': 'fe7c5e8785e0a05833e1bfc3e002ec7f55af21e306b2e7154a448c1f54ccfb0d'}}]}, 20, ['--no-xet'])
    GATES.record("download", fetched_ok, json.dumps(fetched))
# The engine reads its weights from this override; nothing else differs from the Mac's settings.
os.environ["VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"] = "/tmp/omnivoice-base"


In [ ]:
OUT = REPO / "eval/runs/frontend-listening-v1"
RESULT = {}
if not GATES.skip("renders", ("download",)):
    run = run_bounded([sys.executable, "scripts/render_frontend_conditions.py", "--cer"], cwd=REPO,
                      timeout=60 * 60, log=LOGS / "render.log", monitor_every=300)
    RUNS["render"] = run.record()
    print(run.summary())
    manifest_path = OUT / "manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {"renders": [], "events": []}
    rendered = [r for r in manifest["renders"] if r.get("wav") and Path(r["wav"]).is_file()]
    scored = [r for r in rendered if r.get("cer") is not None]
    text = run.log.read_text(errors="replace") if run.log.is_file() else ""
    by_condition = {}
    for r in scored:
        by_condition.setdefault(r["condition"], []).append(r["cer"])
    checked = [e for e in manifest["events"] if e.get("event") == "marks_checked_through_the_text_pipeline"]
    RESULT = {"rendered": len(rendered), "scored": len(scored),
              "worker": [line.strip() for line in text.splitlines() if "OmniVoice ready" in line][:1],
              "mean_cer": {c: round(sum(v) / len(v), 4) for c, v in sorted(by_condition.items())},
              "other_changes_by_the_app": checked[0]["other_changes_by_the_app"] if checked else None}
    GATES.record("renders", run.ok and len(rendered) == 48 and len(scored) == 48 and bool(checked),
                 json.dumps(RESULT, ensure_ascii=False))


In [ ]:
destination = Path("/kaggle/working/frontend-listening-v1")
destination.mkdir(exist_ok=True)
for name in ("manifest.json", "texts.json"):
    if (OUT / name).is_file():
        shutil.copy2(OUT / name, destination / name)
if (OUT / "wavs").is_dir():
    shutil.copytree(OUT / "wavs", destination / "wavs", dirs_exist_ok=True)
(destination / "gates.json").write_text(json.dumps(GATES.summary(), indent=2))
(destination / "runs.json").write_text(json.dumps(RUNS, indent=2))
(destination / "result.json").write_text(json.dumps(RESULT, indent=2, ensure_ascii=False))
shutil.copy2("/kaggle/working/environment.json", destination / "environment.json")
print("gates:", json.dumps(GATES.summary(), indent=1))
